In [6]:
from transformers import PreTrainedTokenizerFast, GPT2LMHeadModel
import torch
import torch.nn.functional as F

# ────────────────────────────────────────────────────────────────────────────────
# 1) KoGPT2 Fast 토크나이저 & 언어모델 로드
# ────────────────────────────────────────────────────────────────────────────────
tokenizer = PreTrainedTokenizerFast.from_pretrained("skt/kogpt2-base-v2")
tokenizer.pad_token = tokenizer.eos_token  # pad_token 설정

gpt2 = GPT2LMHeadModel.from_pretrained(
    "skt/kogpt2-base-v2",
    trust_remote_code=True      # custom model 코드를 허용
)
gpt2.eval()

# ────────────────────────────────────────────────────────────────────────────────
# 2) 감정 프롬프트 준비 (예: “OO이라는 감정을 느끼시는 군요. ” … “라는 상황에서…”)
# ────────────────────────────────────────────────────────────────────────────────
emotion_labels = ["분노","혐오","공포","기쁨","중립","슬픔","놀람"]
prefix_tokens  = [ tokenizer(f"{lab}이라는 감정을 느끼시는 군요. ", return_tensors="pt").input_ids[0]
                   for lab in emotion_labels ]
max_pref      = max(t.size(0) for t in prefix_tokens)
prompt_prefix = torch.stack([
    F.pad(t, (0,max_pref-t.size(0)), value=tokenizer.pad_token_id)
    for t in prefix_tokens
], dim=0)                            # (7, max_pref)

suffix_ids    = tokenizer(
    "라는 상황에서 제가 위로의 말을 건네자면",
    return_tensors="pt"
).input_ids[0]                       # (suffix_len,)

# ────────────────────────────────────────────────────────────────────────────────
# 3) 분류기(TorchScript) & KoGPT2 결합 모델
# ────────────────────────────────────────────────────────────────────────────────
classifier = torch.jit.load("integrated_emotion_model.pt")
classifier.eval()

class IntegratedComfortModel(nn.Module):
    def __init__(self, classifier, gpt2, prompt_pref, suffix, max_gen=50):
        super().__init__()
        self.classifier   = classifier
        self.gpt2         = gpt2
        self.register_buffer("prompt_pref", prompt_pref)
        self.register_buffer("suffix", suffix)
        self.max_gen      = max_gen

    def forward(self, text_ids, image):
        probs   = self.classifier(text_ids, image)      # (1,7)
        emo_id  = torch.argmax(probs, dim=-1)           # (1,)
        prefix  = self.prompt_pref[emo_id]              # (1, max_pref)
        suffix  = self.suffix.unsqueeze(0)               # (1, suffix_len)
        seq_in  = torch.cat([prefix, text_ids, suffix], dim=1)
        out_ids = self.gpt2.generate(
            seq_in,
            max_length=seq_in.size(1)+self.max_gen,
            pad_token_id=tokenizer.pad_token_id,
            do_sample=False
        )
        return out_ids                                  # (1, total_len)

# ────────────────────────────────────────────────────────────────────────────────
# 1) 예시 입력 정의 (트레이스용)
# ────────────────────────────────────────────────────────────────────────────────
vocab_size     = 50000
max_seq_length = 98

# 텍스트: 배치 1, 길이 max_seq_length
example_text  = torch.randint(0, vocab_size, (1, max_seq_length))
# 이미지: 배치 1, 3×224×224
example_image = torch.randn(1, 3, 224, 224)

# ────────────────────────────────────────────────────────────────────────────────
# 2) 모델 인스턴스화
# ────────────────────────────────────────────────────────────────────────────────
model = IntegratedComfortModel(
    classifier   = classifier,
    gpt2         = gpt2,
    prompt_pref  = prompt_prefix,  # 앞에서 만든 prompt_prefix_ids
    suffix       = suffix_ids,
    max_gen      = 50
)
model.eval()

# ────────────────────────────────────────────────────────────────────────────────
# 3) TorchScript Trace
# ────────────────────────────────────────────────────────────────────────────────
# generate() 내부가 실제로 한 번 실행되면서 그 경로의 연산만 기록됩니다.
traced_model = torch.jit.trace(
    model,
    (example_text, example_image),
    strict=False
)
torch.jit.save(traced_model, "integrated_comfort_traced.pt")
print("✅ TorchScript (trace) saved: integrated_comfort_traced.pt")

# ────────────────────────────────────────────────────────────────────────────────
# 4) Core ML 변환
# ────────────────────────────────────────────────────────────────────────────────
import coremltools as ct

mlmodel = ct.convert(
    traced_model,
    source="pytorch",
    inputs=[
        ct.TensorType(name="text_ids",  shape=example_text.shape,  dtype=torch.int64),
        ct.TensorType(name="image",     shape=example_image.shape, dtype=torch.float32),
    ],
    outputs=["output_ids"],
    convert_to="mlprogram",
    minimum_deployment_target=ct.target.iOS16
)
mlmodel.save("IntegratedComfort.mlmodel")
print("✅ Core ML model saved: IntegratedComfort.mlmodel")

The tokenizer class you load from this checkpoint is not the same type as the class this function is called from. It may result in unexpected tokenization. 
The tokenizer class you load from this checkpoint is 'GPT2Tokenizer'. 
The class this function is called from is 'PreTrainedTokenizerFast'.
The argument `trust_remote_code` is to be used with Auto classes. It has no effect here and is ignored.
/home/kangnengyee/.local/lib/python3.8/site-packages/transformers/generation/utils.py:1806: TracerWarning: torch.tensor results are registered as constants in the trace. You can safely ignore this warning if you use this function to create tensors out of constant variables that would be the same every time you call this function. In any other case, this might cause the trace to be incorrect.
  return torch.tensor(token, device=device, dtype=torch.long)
The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attenti

RuntimeError: 0 INTERNAL ASSERT FAILED at "../torch/csrc/jit/ir/alias_analysis.cpp":615, please report a bug to PyTorch. We don't have an op for aten::full but it isn't a special case.  Argument types: int[], bool, int, NoneType, Device, bool, 

Candidates:
	aten::full.names(int[] size, Scalar fill_value, *, str[]? names, ScalarType? dtype=None, Layout? layout=None, Device? device=None, bool? pin_memory=None) -> Tensor
	aten::full(SymInt[] size, Scalar fill_value, *, ScalarType? dtype=None, Layout? layout=None, Device? device=None, bool? pin_memory=None) -> Tensor
	aten::full.names_out(int[] size, Scalar fill_value, *, str[]? names, Tensor(a!) out) -> Tensor(a!)
	aten::full.out(SymInt[] size, Scalar fill_value, *, Tensor(a!) out) -> Tensor(a!)

In [ ]:
coreml따로 저장 후 swift코

In [12]:
import coremltools as ct, numpy as np, torch

classifier_traced = torch.jit.load("integrated_emotion_model.pt")

ml_classifier_nn = ct.convert(
    classifier_traced,
    source="pytorch",
    inputs=[
        ct.TensorType(name="text_ids", shape=(1, 98),   dtype=np.int32),
        ct.TensorType(name="image",    shape=(1, 3,224,224), dtype=np.float32),
    ],
    outputs=["emotion_probs"],
    convert_to="neuralnetwork",    
    minimum_deployment_target=ct.target.iOS14  # ← iOS14 이하로 설정
)

ml_classifier_nn.save("EmotionClassifier.mlmodel")
print("✅ EmotionClassifier.mlmodel (neuralnetwork) 생성 완료")


Support for converting Torch Script Models is experimental. If possible you should use a traced model for conversion.
Core ML embedding (gather) layer does not support any inputs besides the weights and indices. Those given will be ignored.
Translating MIL ==> NeuralNetwork Ops: 100%|████████████████████████████████████████| 114/114 [00:01<00:00, 59.34 ops/s]


✅ EmotionClassifier.mlmodel (neuralnetwork) 생성 완료


In [15]:
import coremltools as ct
import numpy as np
import torch

# 1) TorchScript로 저장된 모듈 로드
traced_next = torch.jit.load("GPT2NextToken.pt")

# 2) Core ML 변환 (neuralnetwork 포맷, iOS14 타깃)
ml_next_nn = ct.convert(
    traced_next,
    source="pytorch",
    inputs=[
        ct.TensorType(name="input_ids",      shape=(1,100), dtype=np.int32),
        ct.TensorType(name="attention_mask", shape=(1,100), dtype=np.int32),
    ],
    outputs=["next_token_logits"],
    convert_to="neuralnetwork",          # ← neuralnetwork 로 변경
    minimum_deployment_target=ct.target.iOS14
)

# 3) .mlmodel 로 저장
ml_next_nn.save("GPT2NextToken.mlmodel")
print("✅ Saved single-file Core ML model: GPT2NextToken.mlmodel")


Support for converting Torch Script Models is experimental. If possible you should use a traced model for conversion.
Saving value type of int64 into a builtin type of int32, might lose precision!               | 0/1040 [00:00<?, ? ops/s]
Saving value type of int64 into a builtin type of int32, might lose precision!
Translating MIL ==> NeuralNetwork Ops: 100%|████████████████████████████████████████| 852/852 [00:11<00:00, 73.87 ops/s]


✅ Saved single-file Core ML model: GPT2NextToken.mlmodel


In [ ]:
mlpackage로 저장

In [ ]:
import coremltools as ct, numpy as np, torch

classifier_traced = torch.jit.load("integrated_emotion_model.pt")

ml_classifier = ct.convert(
    classifier_traced,
    source="pytorch",
    inputs=[
        ct.TensorType(name="text_ids", shape=(1, 98),   dtype=np.int32),
        ct.TensorType(name="image",    shape=(1, 3,224,224), dtype=np.float32),
    ],
    outputs=["emotion_probs"],
    convert_to="mlprogram",    
    minimum_deployment_target=ct.target.iOS16  # iOS15 이상
)

ml_classifier.save("EmotionClassifier.mlpackage")  # ← .mlpackage
print("✅ EmotionClassifier.mlpackage (mlprogram) 생성 완료")


In [14]:
from transformers import PreTrainedTokenizerFast, GPT2LMHeadModel
import torch
import torch.nn as nn
import torch.nn.functional as F
import coremltools as ct
import numpy as np

# 1) 토크나이저 & GPT2LMHeadModel 로드
tokenizer = PreTrainedTokenizerFast.from_pretrained("skt/kogpt2-base-v2")
tokenizer.pad_token = tokenizer.eos_token

gpt2 = GPT2LMHeadModel.from_pretrained(
    "skt/kogpt2-base-v2",
    trust_remote_code=True
).eval()

# 2) 다음 토큰 예측용 래퍼 정의
class GPT2NextToken(nn.Module):
    def __init__(self, gpt2_model):
        super().__init__()
        self.gpt2 = gpt2_model

    def forward(self,
                input_ids: torch.LongTensor,       # (batch, seq_len)
                attention_mask: torch.LongTensor   # (batch, seq_len)
    ) -> torch.FloatTensor:
        logits = self.gpt2(input_ids=input_ids,
                           attention_mask=attention_mask).logits
        # 마지막 위치의 로짓만 반환
        return logits[:, -1, :]  # (batch, vocab_size)

# 3) 래퍼 인스턴스 생성 & Trace 저장
gpt2_next = GPT2NextToken(gpt2).eval()

# 예시 입력 (배치=1, 길이=100)
example_ids  = torch.randint(0, tokenizer.vocab_size,       (1,100), dtype=torch.long)
example_mask = torch.ones_like(example_ids, dtype=torch.long)

traced_next = torch.jit.trace(
    gpt2_next,
    (example_ids, example_mask),
    strict=False
)
# TorchScript 모델 저장
traced_next.save("GPT2NextToken.pt")
print("✅ Saved TorchScript: GPT2NextToken.pt")

# 4) Core ML 변환 (.mlpackage)
ml_next = ct.convert(
    traced_next,
    source="pytorch",
    inputs=[
        ct.TensorType(name="input_ids",      shape=example_ids.shape,  dtype=np.int32),
        ct.TensorType(name="attention_mask", shape=example_mask.shape, dtype=np.int32),
    ],
    outputs=["next_token_logits"],
    convert_to="mlprogram",               # iOS15+ 권장 포맷
    minimum_deployment_target=ct.target.iOS16
)
ml_next.save("GPT2NextToken.mlpackage")
print("✅ Saved Core ML Program: GPT2NextToken.mlpackage")


The tokenizer class you load from this checkpoint is not the same type as the class this function is called from. It may result in unexpected tokenization. 
The tokenizer class you load from this checkpoint is 'GPT2Tokenizer'. 
The class this function is called from is 'PreTrainedTokenizerFast'.
The argument `trust_remote_code` is to be used with Auto classes. It has no effect here and is ignored.


✅ Saved TorchScript: GPT2NextToken.pt


Saving value type of int64 into a builtin type of int32, might lose precision!               | 0/1046 [00:00<?, ? ops/s]
Saving value type of int64 into a builtin type of int32, might lose precision!
Running MIL frontend_pytorch pipeline: 100%|█████████████████████████████████████████| 5/5 [00:00<00:00, 55.53 passes/s]
/home/kangnengyee/.local/lib/python3.8/site-packages/coremltools/converters/mil/mil/ops/defs/iOS15/elementwise_unary.py:894: RuntimeWarning: overflow encountered in cast
  return input_var.val.astype(dtype=string_to_nptype(dtype_val))
Running MIL backend_mlprogram pipeline: 100%|██████████████████████████████████████| 12/12 [00:00<00:00, 54.95 passes/s]


✅ Saved Core ML Program: GPT2NextToken.mlpackage


In [ ]:
swift code

In [ ]:
import CoreML
import UIKit

// MARK: ———— 모델 로드 —————————————————————————————————————————————————————
let clsConfig = MLModelConfiguration()
let emotionClassifier = try! EmotionClassifier(configuration: clsConfig)
let nextTokenModel     = try! GPT2NextToken(configuration: clsConfig)

// MARK: ———— 토크나이저/상수 정의 ——————————————————————————————————————————————
let tokenizer = Kogpt2SwiftTokenizer()  // Swift용 KoGPT2 토크나이저 래퍼
let padTokenID: Int32 = tokenizer.padTokenID
let eosTokenID: Int32 = tokenizer.eosTokenID

let emotionLabels = ["분노","혐오","공포","기쁨","중립","슬픔","놀람"]
let maxSeqLen     = 98
let maxGenLen     = 50

// MLMultiArray 생성 헬퍼
func makeMLMultiArray(_ data: [Int32], shape: [NSNumber]) -> MLMultiArray {
    let arr = try! MLMultiArray(shape: shape, dataType: .int32)
    for (i, v) in data.enumerated() {
        arr[i] = NSNumber(value: v)
    }
    return arr
}

// MARK: ———— 메인: 위로 메시지 생성 함수 ——————————————————————————————————————————
func generateComfortMessage(text: String, image: UIImage) throws -> String {
    // 1) 이미지 → CVPixelBuffer
    guard let pixelBuffer = image.toCVPixelBuffer() else {
        throw NSError(domain:"", code:-1, userInfo:[NSLocalizedDescriptionKey:"버퍼 변환 실패"])
    }

    // 2) 원문 토큰화 + 패딩
    var textIDs = tokenizer.encode(text)                     // [Int32]
    textIDs = Array(textIDs.prefix(maxSeqLen))               // 자르고
    textIDs += Array(repeating: padTokenID, count: maxSeqLen - textIDs.count)

    let textArray = makeMLMultiArray(textIDs, shape: [1, NSNumber(value:maxSeqLen)])

    // 3) 감정 분류기 호출
    let clsOut = try emotionClassifier.prediction(
        text_ids: textArray,
        image:    pixelBuffer
    )
    let probs = clsOut.emotion_probs   // MLMultiArray of Doubles, length=7
    // argmax
    let emoIndex = (0..<probs.count).max { probs[$0].doubleValue < probs[$1].doubleValue }!

    // 4) 템플릿 조합
    let prefix = "\(emotionLabels[emoIndex])이라는 감정을 느끼시는 군요. "
    let suffix = "라는 상황에서 제가 위로의 말을 건네자면"
    var fullText = prefix + text + suffix

    // 5) 시퀀스 다시 토큰화 (자동 완성을 위한 입력)
    var seqIDs = tokenizer.encode(fullText)                 // [Int32]
    var attentionMask = seqIDs.map { $0 == padTokenID ? 0 : 1 }

    // 6) 한 스텝씩 next‑token 호출
    for _ in 0..<maxGenLen {
        let seqLen = seqIDs.count
        let inputArr = makeMLMultiArray(seqIDs,       shape: [1, NSNumber(value:seqLen)])
        let maskArr  = makeMLMultiArray(attentionMask, shape: [1, NSNumber(value:seqLen)])

        let nextOut  = try nextTokenModel.prediction(
            input_ids:      inputArr,
            attention_mask: maskArr
        )
        let logits = nextOut.next_token_logits        // MLMultiArray length=vocab_size

        // logits → Swift [Double]
        let ptr = logits.dataPointer.bindMemory(to: Double.self, capacity: logits.count)
        let buf = UnsafeBufferPointer(start: ptr, count: logits.count)
        guard let nextId = buf.enumerated().max(by: { $0.element < $1.element })?.offset else {
            break
        }
        if Int32(nextId) == eosTokenID { break }

        seqIDs.append(Int32(nextId))
        attentionMask.append(1)
    }

    // 7) 최종 디코딩
    return tokenizer.decode(seqIDs)
}


In [ ]:
토크나이저

In [1]:
!pip install transformers



Defaulting to user installation because normal site-packages is not writeable


In [2]:
from transformers import PreTrainedTokenizerFast

# 1) KoGPT2 pretrained 토크나이저 불러오기
tokenizer = PreTrainedTokenizerFast.from_pretrained("skt/kogpt2-base-v2")

# 2) 로컬 폴더에 저장
save_dir = "./kogpt2_tokenizer"
tokenizer.save_pretrained(save_dir)

print("Saved vocab and merges to", save_dir)



The tokenizer class you load from this checkpoint is not the same type as the class this function is called from. It may result in unexpected tokenization. 
The tokenizer class you load from this checkpoint is 'GPT2Tokenizer'. 
The class this function is called from is 'PreTrainedTokenizerFast'.


Saved vocab and merges to ./kogpt2_tokenizer
